In [1]:
%pip install pandas
import pandas as pd

df = pd.read_csv("customer_shopping_behavior.csv")

df.head()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [3]:
df['Review Rating'].fillna(df['Review Rating'].median(), inplace=True)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9504\4050516494.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Review Rating'].fillna(df['Review Rating'].median(), inplace=True)


0       3.1
1       3.1
2       3.1
3       3.5
4       2.7
       ... 
3895    4.2
3896    4.5
3897    2.9
3898    3.8
3899    3.1
Name: Review Rating, Length: 3900, dtype: float64

In [4]:
df['Age Group'] = pd.cut(df['Age'],
                        bins=[18,25,35,50,70],
                        labels=['Young Adult','Adult','Middle Age','Senior'])

In [5]:
df['Discount Flag'] = df['Discount Applied'].map({'Yes':1,'No':0})

In [6]:
df['High Value'] = df['Purchase Amount (USD)'].apply(lambda x: 1 if x > 80 else 0)

In [7]:
def segment(x):
    if x == 1:
        return "New"
    elif x <= 5:
        return "Returning"
    else:
        return "Loyal"

df['Customer Segment'] = df['Previous Purchases'].apply(segment)

In [8]:
df['Frequency Category'] = pd.cut(df['Previous Purchases'],
                                 bins=[0,2,5,20],
                                 labels=['Low','Medium','High'])

In [9]:
df.to_csv("cleaned_data.csv", index=False)

In [10]:
df['CLV'] = df['Purchase Amount (USD)'] * df['Previous Purchases']

In [11]:
df['AOV'] = df['Purchase Amount (USD)'] / (df['Previous Purchases'] + 1)

In [12]:
df['Discount Impact'] = df['Purchase Amount (USD)'] * df['Discount Flag']

In [13]:
def review_category(x):
    if x >= 4:
        return "Good"
    elif x >= 3:
        return "Average"
    else:
        return "Poor"

df['Review Category'] = df['Review Rating'].apply(review_category)

In [14]:
df.to_csv("cleaned_data_advanced.csv", index=False)

In [15]:
!pip install pymysql sqlalchemy


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
pip install sqlalchemy pymysql pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
from sqlalchemy import create_engine

In [17]:
df = pd.read_csv("cleaned_data_advanced.csv")

In [21]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load your cleaned dataset
df = pd.read_csv("cleaned_data_advanced.csv")

# MySQL credentials
username = "root"
password = quote_plus("Guru@2503")
host = "localhost"
port = "3306"
database = "customer_behavior"

# Create connection
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Test connection (optional but recommended)
engine.connect()

# Upload data to MySQL
df.to_sql("customer", engine, if_exists="replace", index=False)

# Verify data
pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Frequency of Purchases,Age Group,Discount Flag,High Value,Customer Segment,Frequency Category,CLV,AOV,Discount Impact,Review Category
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Fortnightly,Senior,1,0,Loyal,High,742,3.533333,53,Average
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Fortnightly,Young Adult,1,0,Returning,Low,128,21.333333,64,Average
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Weekly,Middle Age,1,0,Loyal,NaN,1679,3.041667,73,Average
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Weekly,Young Adult,1,1,Loyal,NaN,4410,1.800000,90,Average
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Annually,Middle Age,1,0,Loyal,NaN,1519,1.531250,49,Poor


In [22]:
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('(', '')
df.columns = df.columns.str.replace(')', '')
df.columns = df.columns.str.replace('-', '_')